# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

## Setup: load and clean the data

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Standardize column names: lowercase, underscores instead of spaces
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.drop(columns=[c for c in df.columns if c.startswith("unnamed")])

# Make sure we have a month column (derived from the policy date if missing)
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")
if "month" not in df.columns:
    df["month"] = df["effective_to_date"].dt.month

print(df.shape)
df.head()

In [ ]:
# Quick check of missing values in the columns we'll use
df[["response", "total_claim_amount", "monthly_premium_auto", "customer_lifetime_value",
    "policy_type", "gender", "state", "education", "sales_channel", "month"]].isna().sum()

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [ ]:
low_claim_yes = df[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")]

print(f"{len(low_claim_yes)} customers out of {len(df)}")
low_claim_yes.head()

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [ ]:
yes = df[df["response"] == "Yes"]

yes_summary = (
    yes.groupby(["policy_type", "gender"])
       .agg(avg_monthly_premium=("monthly_premium_auto", "mean"),
            avg_clv=("customer_lifetime_value", "mean"),
            avg_claim=("total_claim_amount", "mean"),
            n_customers=("customer", "count"))
       .round(2)
)

# Premium collected per $ of claims: higher = more profitable / lower risk
yes_summary["premium_to_claim_ratio"] = (
    yes_summary["avg_monthly_premium"] / yes_summary["avg_claim"]
).round(3)

yes_summary.sort_values("premium_to_claim_ratio", ascending=False)

In [ ]:
# Same view for ALL customers, to see whether responders differ from the overall base
df.groupby(["policy_type", "gender"])[["monthly_premium_auto", "customer_lifetime_value",
                                        "total_claim_amount"]].mean().round(2)

**Discussion (update with your actual numbers):**

- The `premium_to_claim_ratio` column is the key comparison: a segment that pays a high monthly premium but has a relatively low average claim is the most profitable. A segment whose average claim is high relative to what it pays is the riskiest.
- Look at the top and bottom rows of the sorted table. Name the policy type/gender combinations at each end and note whether the gap comes from premiums (revenue side) or from claims (cost side).
- Check `n_customers`: segments with very few responders (Special Auto often has few) give unstable averages, so don't draw strong conclusions from them.
- Personal Auto usually holds the large majority of customers, so even a small edge in its ratio matters more to the business than a big edge in a tiny segment.
- A high `avg_clv` with a good ratio marks a segment that is both valuable over time and low risk, which is the best target for future campaigns.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [ ]:
customers_by_state = df.groupby("state")["customer"].nunique().sort_values(ascending=False)
print(customers_by_state, "\n")

big_states = customers_by_state[customers_by_state > 500]
big_states

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [ ]:
clv_stats = (
    df.groupby(["education", "gender"])["customer_lifetime_value"]
      .agg(["max", "min", "median"])
      .round(2)
)
clv_stats

In [ ]:
# Side-by-side view of the medians makes gender comparisons easier
clv_stats["median"].unstack("gender")

**Conclusions (update with your actual numbers):**

- CLV is heavily right-skewed: the maximum in each group is many times the median, so a handful of very valuable customers pull the mean up. That is why the median is the fairer measure of a "typical" customer here.
- The minimums are fairly similar across all groups, meaning every segment has low-value customers; the differences are mostly at the top end.
- Compare the medians across education levels: check whether higher education (Master, Doctor) goes with a higher typical CLV, or whether the differences are small.
- Compare F vs M within each education level: note whether one gender is consistently higher or whether it varies by level.
- Groups with fewer customers (usually Doctor) have less reliable max/min values, since one extreme customer can decide them.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [ ]:
policies_by_state_month = df.pivot_table(
    index="state", columns="month", values="policy", aggfunc="count", fill_value=0
)
policies_by_state_month

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [ ]:
# 1. Group by state and month and count policies
state_month = df.groupby(["state", "month"])["policy"].count().reset_index(name="policies_sold")

# 2. Top 3 states by total policies sold
top3 = (state_month.groupby("state")["policies_sold"].sum()
                   .sort_values(ascending=False).head(3).index)
print("Top 3 states:", list(top3))

# 3. Monthly counts for those states only
top3_by_month = (state_month[state_month["state"].isin(top3)]
                 .pivot(index="state", columns="month", values="policies_sold")
                 .loc[top3])
top3_by_month

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [ ]:
# Count responses per channel (rows = channel, columns = Yes/No)
counts = (df.dropna(subset=["response"])
            .pivot_table(index="sales_channel", columns="response",
                         values="customer", aggfunc="count", fill_value=0)
            .reset_index())

# Melt to long format: one row per (channel, response)
long = counts.melt(id_vars="sales_channel", var_name="response", value_name="n")
long

In [ ]:
# Response rate = Yes / total, per channel
totals = long.groupby("sales_channel")["n"].sum()
yes_n = long[long["response"] == "Yes"].set_index("sales_channel")["n"]

response_rate = (yes_n / totals * 100).round(2).sort_values(ascending=False)
response_rate.rename("response_rate_%").to_frame()

**Conclusion:** The channel at the top of this table converts best. Compare it with the lowest channel: if the gap is several percentage points, the marketing team should shift effort toward the stronger channel, or study what it does differently.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9